<a href="https://colab.research.google.com/github/Albonire/clasificador-imagenes-openmp-cuda/blob/main/etapa2_cuda/gpu_model/model-gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU (CUDA) — Clasificador de Somnolencia (MLP)
**Programacion Paralela y Computacion Distribuida · Universidad de Pamplona · 2026-I**

Este notebook **compila, ejecuta y mide** el entrenamiento de la misma red de
`etapa2_cuda/cpu_baseline/base-model-cpu.ipynb`, pero usando **kernels CUDA escritos
a mano** (`train_gpu.cu`) en lugar de TensorFlow. Todo el entrenamiento (forward,
backward, actualizacion de pesos) ocurre en la GPU; este notebook solo orquesta:
compila el `.cu`, lo ejecuta como subproceso, parsea su salida y genera las
graficas/reportes.

**Requiere runtime de Colab con GPU** (Entorno de ejecucion > Cambiar tipo de
entorno de ejecucion > GPU).

**Arquitectura (igual que la version CPU):**

```
Entrada(4096) -> Densa(oculta, ReLU) -> Densa(1, Sigmoide)
```

- **Perdida:** Binary Cross-Entropy (BCE)
- **Optimizador:** SGD escrito a mano (`peso -= tasa_aprendizaje * gradiente / N`)
- **Modo de entrenamiento:** full-batch, igual que la version CPU, para que el
  speedup sea una comparacion 1:1.

**Contenido:**
1. Configuracion (Colab, verificacion de GPU, rutas)
2. Compilacion del kernel CUDA
3. Diseno de los kernels (`train_gpu.cu`)
4. Funcion auxiliar para ejecutar el entrenamiento
5. Busqueda de hiperparametros (epocas 20-50, lr 0.01-0.1, ocultas 64-128)
6. Entrenamiento final en GPU (tiempo medido)
7. Efecto del tamano de bloque (16x16 vs 32x32)
8. Monitoreo de GPU con `nvidia-smi`
9. Evaluacion en test
10. Exportacion de pesos
11. Reporte (`gpu_report.md`, incluye speedup vs. CPU)

---
## 1. Configuración

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Albonire/clasificador-imagenes-openmp-cuda.git"
REPO_DIR = "/content/clasificador-imagenes-openmp-cuda"

if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

if IN_COLAB:
    os.chdir(os.path.join(REPO_DIR, "etapa2_cuda", "gpu_model"))

print(f"En Colab: {IN_COLAB}")
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
!nvidia-smi

In [ ]:
import datetime
import itertools
import re
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

SEED = 42
np.random.seed(SEED)

print("Librerias cargadas.")

In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.exists(os.path.join(REPO_ROOT, "dataset")):
    REPO_ROOT = os.getcwd()

PATHS = {
    "procesado": os.path.join(REPO_ROOT, "dataset", "procesado"),
    "evidencias": os.path.join(REPO_ROOT, "reporte", "evidencias"),
    "gpu_model": os.path.join(REPO_ROOT, "etapa2_cuda", "gpu_model"),
    "cpu_baseline": os.path.join(REPO_ROOT, "etapa2_cuda", "cpu_baseline"),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  {'OK' if os.path.isdir(path) else 'WARN'} {name}: {path}")

TRAIN_CSV = os.path.join(PATHS["procesado"], "train.csv")
VAL_CSV = os.path.join(PATHS["procesado"], "val.csv")
TEST_CSV = os.path.join(PATHS["procesado"], "test.csv")

SOURCE_CU = os.path.join(PATHS["gpu_model"], "train_gpu.cu")
TRAIN_BIN = os.path.join(PATHS["gpu_model"], "train_gpu")
TMP_WEIGHTS = os.path.join(PATHS["gpu_model"], "_tmp_weights.bin")

print(f"\nTrain CSV: {TRAIN_CSV}")
print(f"Val CSV:   {VAL_CSV}")
print(f"Test CSV:  {TEST_CSV}")
print(f"Fuente CUDA: {SOURCE_CU}")

---
## 2. Compilacion del kernel CUDA

`train_gpu.cu` es un programa de linea de comandos independiente (igual que
`etapa1_openmp/preprocess_serial.c`): este notebook solo lo compila con `nvcc` y
lo ejecuta como subproceso para cada combinacion de hiperparametros.

In [ ]:
!nvcc --version
print()
!nvcc -O3 -o "{TRAIN_BIN}" "{SOURCE_CU}"

assert os.path.exists(TRAIN_BIN), "La compilacion fallo: no se genero el binario train_gpu"
print(f"\nCompilado correctamente: {TRAIN_BIN}")

---
## 3. Diseño de los kernels CUDA

Cada pieza de la red tiene un kernel propio en `train_gpu.cu` (ver el archivo para
el codigo completo):

| Pieza de la red | Kernel(s) en `train_gpu.cu` |
|---|---|
| Densa (forward, matriz-matriz) | `matmul_ab_tiled` (con memoria compartida, `block_size` configurable en runtime) |
| Bias + ReLU (capa oculta) | `bias_relu_forward` |
| Sigmoide (capa de salida) | `bias_sigmoid_forward` |
| Perdida BCE | `bce_loss_kernel` (reduccion con `atomicAdd`) |
| Gradiente de la capa de salida | `output_gradient_kernel` (`dZ2 = y_hat - y`) |
| Gradiente de la capa oculta | `hidden_gradient_kernel` (`dZ1 = (dZ2 @ W2^T) * relu'(Z1)`) |
| Gradientes de pesos (`dW1`, `dW2`) | `matmul_atb` (GEMM transpuesto `A^T @ B`) |
| Gradientes de bias (`db1`, `db2`) | `bias_gradient_kernel` |
| Actualizacion SGD | `sgd_update_kernel` (`peso -= lr * grad / N`) |

**Notas de diseño:**
- `matmul_ab_tiled` es el kernel que se usa para el forward de ambas capas densas
  (la operacion mas pesada y mas paralelizable, segun la guia) y es el que se
  prueba con distintos `block_size` (16x16, 32x32) en la seccion 7.
- `matmul_atb` (gradientes de pesos) usa un bloque fijo de 16x16: no es el foco
  del experimento de tamaño de bloque, que se centra en el forward.
- El entrenamiento es **full-batch** (todo el train set por epoca), igual que la
  version CPU, para que el speedup sea una comparacion valida.
- Los pesos se inicializan con Glorot/Xavier uniforme y `srand(42)`, para que la
  inicializacion sea reproducible (aunque no sea bit-a-bit identica a la
  inicializacion de Keras en la version CPU).

---
## 4. Funcion auxiliar para ejecutar el entrenamiento

In [ ]:
def run_training(hidden_units, learning_rate, epochs, block_size, weights_out,
                  monitor_gpu=False, poll_interval=0.1):
    cmd = [
        TRAIN_BIN,
        TRAIN_CSV,
        VAL_CSV,
        str(hidden_units),
        str(learning_rate),
        str(epochs),
        str(block_size),
        weights_out,
    ]

    gpu_samples = []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    if monitor_gpu:
        while proc.poll() is None:
            sample = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True,
            )
            if sample.returncode == 0 and sample.stdout.strip():
                util_str, mem_used_str, mem_total_str = sample.stdout.strip().split(",")
                gpu_samples.append({
                    "t": time.time(),
                    "util_pct": float(util_str),
                    "mem_used_mb": float(mem_used_str),
                    "mem_total_mb": float(mem_total_str),
                })
            time.sleep(poll_interval)

    stdout, stderr = proc.communicate()

    if proc.returncode != 0:
        raise RuntimeError(f"train_gpu fallo (codigo {proc.returncode}):\n{stderr}")

    info = {}
    epoch_history = []
    summary = None
    h2d_seconds = None

    for line in stdout.splitlines():
        parts = line.strip().split(",")
        if not parts or parts[0] == "":
            continue
        tag = parts[0]
        if tag == "INFO":
            info[parts[1]] = parts[2]
        elif tag == "TRANSFER":
            h2d_seconds = float(parts[1])
        elif tag == "EPOCH":
            epoch_history.append({
                "epoch": int(parts[1]),
                "train_loss": float(parts[2]),
                "val_loss": float(parts[3]),
                "val_accuracy": float(parts[4]),
            })
        elif tag == "SUMMARY":
            summary = {
                "hidden_units": int(parts[1]),
                "learning_rate": float(parts[2]),
                "epochs": int(parts[3]),
                "block_size": int(parts[4]),
                "train_loop_seconds": float(parts[5]),
                "h2d_seconds": float(parts[6]),
                "final_train_loss": float(parts[7]),
                "final_val_loss": float(parts[8]),
                "final_val_accuracy": float(parts[9]),
            }

    if summary is None:
        raise RuntimeError(f"No se recibio linea SUMMARY de train_gpu.\nstdout:\n{stdout}\nstderr:\n{stderr}")

    return {
        "info": info,
        "h2d_seconds": h2d_seconds,
        "epoch_history": epoch_history,
        "summary": summary,
        "gpu_samples": gpu_samples,
        "stdout": stdout,
        "stderr": stderr,
    }


print("run_training() listo.")

---
## 5. Busqueda de hiperparametros (GPU)

Misma grilla que la version CPU (extremos de cada rango sugerido por la guia):
epocas 20-50, tasa de aprendizaje 0.01-0.1, neuronas ocultas 64-128.
`block_size` se fija en 32 durante esta busqueda (su efecto se mide aparte en la
seccion 7); los pesos de estas corridas no se conservan (`weights_out` apunta a
un archivo temporal).

In [ ]:
HIDDEN_GRID = [64, 128]
LR_GRID = [0.01, 0.1]
EPOCHS_GRID = [20, 50]
GRID_BLOCK_SIZE = 32

gpu_results = []

for hidden_units, learning_rate, epochs in itertools.product(HIDDEN_GRID, LR_GRID, EPOCHS_GRID):
    result = run_training(hidden_units, learning_rate, epochs, GRID_BLOCK_SIZE, TMP_WEIGHTS)
    summary = result["summary"]
    gpu_results.append({
        "hidden_units": hidden_units,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "train_loop_seconds": summary["train_loop_seconds"],
        "val_loss": summary["final_val_loss"],
        "val_accuracy": summary["final_val_accuracy"],
    })
    print(
        f"hidden={hidden_units:3d} lr={learning_rate:<5} epochs={epochs:3d} "
        f"-> val_acc={summary['final_val_accuracy']:.4f} "
        f"val_loss={summary['final_val_loss']:.4f} "
        f"time={summary['train_loop_seconds']:.4f}s"
    )

gpu_results_df = pd.DataFrame(gpu_results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
gpu_results_df

In [ ]:
best_gpu = gpu_results_df.iloc[0]
BEST_HIDDEN = int(best_gpu["hidden_units"])
BEST_LR = float(best_gpu["learning_rate"])
BEST_EPOCHS = int(best_gpu["epochs"])

print("Mejor combinacion (mayor accuracy en validacion):")
print(f"  hidden_units  = {BEST_HIDDEN}")
print(f"  learning_rate = {BEST_LR}")
print(f"  epochs        = {BEST_EPOCHS}")
print(f"  val_accuracy  = {best_gpu['val_accuracy']:.4f}")
print(f"  val_loss      = {best_gpu['val_loss']:.4f}")

---
## 6. Entrenamiento final en GPU (tiempo medido)

Se reentrena desde cero con la mejor combinacion, `block_size=32`, y se guardan
los pesos finales. `GPU_TRAIN_TIME_SEC` (`train_loop_seconds` del programa CUDA,
es decir solo el bucle de epocas, sin contar la transferencia inicial
host->device) es el numero que se compara con `CPU_TRAIN_TIME_SEC` del
notebook CPU para calcular el speedup.

In [ ]:
FINAL_BLOCK_SIZE = 32
WEIGHTS_BIN_PATH = os.path.join(PATHS["gpu_model"], "weights_gpu.bin")

final_result = run_training(BEST_HIDDEN, BEST_LR, BEST_EPOCHS, FINAL_BLOCK_SIZE, WEIGHTS_BIN_PATH)
final_summary = final_result["summary"]

GPU_TRAIN_TIME_SEC = final_summary["train_loop_seconds"]
H2D_SECONDS = final_result["h2d_seconds"]
GPU_NAME = final_result["info"].get("gpu_name", "desconocida")

print(f"GPU detectada: {GPU_NAME}")
print(f"Entrenamiento final: hidden={BEST_HIDDEN}, lr={BEST_LR}, epochs={BEST_EPOCHS}, block_size={FINAL_BLOCK_SIZE}")
print(f"Transferencia host->device: {H2D_SECONDS:.6f} s")
print(f"Tiempo de entrenamiento en GPU (bucle de epocas): {GPU_TRAIN_TIME_SEC:.6f} s")
print(f"Val accuracy final: {final_summary['final_val_accuracy']:.4f}")
print(f"Pesos guardados (binario): {WEIGHTS_BIN_PATH}")

In [ ]:
final_history_df = pd.DataFrame(final_result["epoch_history"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(final_history_df["epoch"], final_history_df["train_loss"], label="train")
axes[0].plot(final_history_df["epoch"], final_history_df["val_loss"], label="val")
axes[0].set_title("Perdida BCE (GPU)")
axes[0].set_xlabel("Epoca")
axes[0].legend()

axes[1].plot(final_history_df["epoch"], final_history_df["val_accuracy"], label="val accuracy")
axes[1].set_title("Accuracy en validacion (GPU)")
axes[1].set_xlabel("Epoca")
axes[1].legend()

plt.tight_layout()
curves_path = os.path.join(PATHS["evidencias"], "gpu_training_curves.png")
fig.savefig(curves_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {curves_path}")

---
## 7. Efecto del tamaño de bloque

Se compara `block_size=16` contra `block_size=32` usando los mismos
hiperparametros finales. El valor de 32x32 ya se midio en la seccion anterior
(`GPU_TRAIN_TIME_SEC`); aqui solo se ejecuta una corrida adicional con
`block_size=16` (sus pesos no se conservan).

In [ ]:
BLOCK_SIZES = [16, 32]
block_size_results = []

for block_size in BLOCK_SIZES:
    if block_size == FINAL_BLOCK_SIZE:
        summary = final_summary
    else:
        result_bs = run_training(BEST_HIDDEN, BEST_LR, BEST_EPOCHS, block_size, TMP_WEIGHTS)
        summary = result_bs["summary"]

    block_size_results.append({
        "block_size": block_size,
        "train_loop_seconds": summary["train_loop_seconds"],
        "final_val_accuracy": summary["final_val_accuracy"],
    })
    print(f"block_size={block_size:2d}x{block_size:<2d} -> tiempo={summary['train_loop_seconds']:.6f}s")

block_size_df = pd.DataFrame(block_size_results)
block_size_df

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(block_size_df["block_size"].astype(str), block_size_df["train_loop_seconds"], color=["#4C72B0", "#DD8452"])
ax.set_xlabel("block_size (NxN)")
ax.set_ylabel("Tiempo de entrenamiento (s)")
ax.set_title("Efecto del tamaño de bloque en el tiempo de entrenamiento")

plt.tight_layout()
block_size_plot_path = os.path.join(PATHS["evidencias"], "gpu_block_size_effect.png")
fig.savefig(block_size_plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {block_size_plot_path}")

---
## 8. Monitoreo de GPU con `nvidia-smi`

El modelo y el dataset de este proyecto son pequeños (4096 entradas, ~935
muestras de entrenamiento, una sola capa oculta), por lo que el entrenamiento
real (seccion 6) puede terminar en fracciones de segundo en una GPU moderna —
muy poco tiempo para que `nvidia-smi` capture varias muestras representativas.

Para esta medicion especifica se usa un numero de epocas inflado
(`MONITOR_EPOCHS`), **solo para alargar la ventana de muestreo**; no afecta los
pesos exportados ni el tiempo reportado como `GPU_TRAIN_TIME_SEC`.

In [ ]:
MONITOR_EPOCHS = max(BEST_EPOCHS, 300)
POLL_INTERVAL_SEC = 0.05

monitor_result = run_training(
    BEST_HIDDEN, BEST_LR, MONITOR_EPOCHS, FINAL_BLOCK_SIZE, TMP_WEIGHTS,
    monitor_gpu=True, poll_interval=POLL_INTERVAL_SEC,
)

gpu_samples = monitor_result["gpu_samples"]
print(f"Muestras de nvidia-smi capturadas: {len(gpu_samples)}")

if gpu_samples:
    samples_df = pd.DataFrame(gpu_samples)
    samples_df["t_rel"] = samples_df["t"] - samples_df["t"].iloc[0]

    GPU_UTIL_MAX = samples_df["util_pct"].max()
    GPU_UTIL_AVG = samples_df["util_pct"].mean()
    GPU_MEM_PEAK_MB = samples_df["mem_used_mb"].max()
    GPU_MEM_TOTAL_MB = samples_df["mem_total_mb"].iloc[0]

    print(f"Utilizacion GPU: max={GPU_UTIL_MAX:.1f}%  promedio={GPU_UTIL_AVG:.1f}%")
    print(f"Memoria GPU: pico={GPU_MEM_PEAK_MB:.0f} MB de {GPU_MEM_TOTAL_MB:.0f} MB")
else:
    samples_df = pd.DataFrame(columns=["t_rel", "util_pct", "mem_used_mb"])
    GPU_UTIL_MAX = GPU_UTIL_AVG = GPU_MEM_PEAK_MB = GPU_MEM_TOTAL_MB = float("nan")
    print("No se capturaron muestras: el entrenamiento termino antes del primer sondeo de nvidia-smi.")

In [ ]:
if not samples_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(samples_df["t_rel"], samples_df["util_pct"], marker="o")
    axes[0].set_title("Utilizacion de GPU durante el entrenamiento")
    axes[0].set_xlabel("Tiempo (s)")
    axes[0].set_ylabel("Utilizacion (%)")

    axes[1].plot(samples_df["t_rel"], samples_df["mem_used_mb"], marker="o", color="#DD8452")
    axes[1].set_title("Memoria de GPU usada")
    axes[1].set_xlabel("Tiempo (s)")
    axes[1].set_ylabel("Memoria (MB)")

    plt.tight_layout()
    monitor_plot_path = os.path.join(PATHS["evidencias"], "gpu_nvidia_smi_monitor.png")
    fig.savefig(monitor_plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  {monitor_plot_path}")
else:
    monitor_plot_path = None
    print("Sin muestras para graficar.")

---
## 9. Evaluacion en test

In [ ]:
def load_gpu_weights(path):
    with open(path, "rb") as f:
        dims = np.fromfile(f, dtype=np.int32, count=2)
        input_dim, hidden_units = int(dims[0]), int(dims[1])
        w1_raw = np.fromfile(f, dtype=np.float32, count=input_dim * hidden_units).reshape(input_dim, hidden_units)
        b1_raw = np.fromfile(f, dtype=np.float32, count=hidden_units)
        w2_raw = np.fromfile(f, dtype=np.float32, count=hidden_units)
        b2_raw = np.fromfile(f, dtype=np.float32, count=1)
    return input_dim, hidden_units, w1_raw, b1_raw, w2_raw, b2_raw


INPUT_DIM, HIDDEN_UNITS, w1_raw, b1_raw, w2_raw, b2_raw = load_gpu_weights(WEIGHTS_BIN_PATH)

# Formato app_streamlit: W1 (oculta, entrada), W2 (1, oculta)
W1 = w1_raw.T
b1 = b1_raw
W2 = w2_raw.reshape(1, HIDDEN_UNITS)
b2 = b2_raw

print(f"W1: {W1.shape}, b1: {b1.shape}, W2: {W2.shape}, b2: {b2.shape}")

In [ ]:
TARGET = "label"


def load_split(path):
    df = pd.read_csv(path)
    feature_cols = [c for c in df.columns if c != TARGET]
    x = df[feature_cols].to_numpy(dtype=np.float32)
    y = df[TARGET].to_numpy(dtype=np.float32)
    return x, y


X_test, y_test = load_split(TEST_CSV)


def forward_numpy(x, w1, b1, w2, b2):
    z1 = x @ w1.T + b1
    a1 = np.maximum(z1, 0)
    z2 = a1 @ w2.T + b2
    y_hat = 1.0 / (1.0 + np.exp(-z2))
    return y_hat.flatten()


y_prob = forward_numpy(X_test, W1, b1, W2, b2)
y_pred = (y_prob >= 0.5).astype(np.int32)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Test accuracy:  {acc:.4f}")
print(f"Test precision: {prec:.4f}")
print(f"Test recall:    {rec:.4f}")
print(f"Test F1:        {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Abiertos (0)", "Cerrados (1)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Greens")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Abiertos", "Cerrados"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Abiertos", "Cerrados"])
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusion - Test (GPU)")
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, str(cm[i, j]), ha="center", va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

plt.tight_layout()
cm_path = os.path.join(PATHS["evidencias"], "gpu_confusion_matrix.png")
fig.savefig(cm_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {cm_path}")

---
## 10. Exportacion de pesos

Mismo formato `.npz` que la version CPU (`W1, b1, W2, b2`), listo para
`app_streamlit/app.py`. Se guarda como checkpoint de esta version GPU, sin
sobrescribir `modelo/weights.npz`.

In [ ]:
weights_npz_path = os.path.join(PATHS["gpu_model"], "weights_gpu.npz")
np.savez(weights_npz_path, W1=W1, b1=b1, W2=W2, b2=b2)

print(f"Pesos guardados: {weights_npz_path}")
print(f"  W1: {W1.shape}, b1: {b1.shape}, W2: {W2.shape}, b2: {b2.shape}")

---
## 11. Reporte

In [ ]:
cpu_report_path = os.path.join(PATHS["cpu_baseline"], "cpu_baseline_report.md")
CPU_TRAIN_TIME_SEC = None

if os.path.exists(cpu_report_path):
    with open(cpu_report_path, "r", encoding="utf-8") as f:
        cpu_report_text = f.read()
    match = re.search(r"Tiempo de entrenamiento:\**\s*([0-9.]+)\s*s", cpu_report_text)
    if match:
        CPU_TRAIN_TIME_SEC = float(match.group(1))

if CPU_TRAIN_TIME_SEC is not None:
    SPEEDUP = CPU_TRAIN_TIME_SEC / GPU_TRAIN_TIME_SEC
    print(f"CPU: {CPU_TRAIN_TIME_SEC:.4f}s | GPU: {GPU_TRAIN_TIME_SEC:.4f}s | Speedup: {SPEEDUP:.2f}x")
else:
    SPEEDUP = None
    print("No se encontro 'cpu_baseline_report.md'; no se pudo calcular el speedup.")

report_path = os.path.join(PATHS["gpu_model"], "gpu_report.md")
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# GPU Report (CUDA) — Clasificador de Somnolencia\n\n")
    f.write(f"**Generado:** {now}\n\n")
    f.write(f"**GPU:** {GPU_NAME}\n\n")

    f.write("## Arquitectura\n\n")
    f.write(f"`Entrada({INPUT_DIM}) -> Densa({BEST_HIDDEN}, ReLU) -> Densa(1, Sigmoide)`\n\n")
    f.write("- **Perdida:** Binary Cross-Entropy\n")
    f.write("- **Optimizador:** SGD escrito a mano (sin momentum)\n")
    f.write("- **Modo de entrenamiento:** full-batch (todo el train set por epoca)\n")
    f.write("- **Kernels:** `matmul_ab_tiled`, `matmul_atb`, `bias_relu_forward`, "
            "`bias_sigmoid_forward`, `bce_loss_kernel`, `output_gradient_kernel`, "
            "`hidden_gradient_kernel`, `bias_gradient_kernel`, `sgd_update_kernel` "
            "(ver `train_gpu.cu`)\n\n")

    f.write("## Busqueda de hiperparametros (GPU, block_size=32)\n\n")
    f.write("| hidden_units | learning_rate | epochs | train_loop_seconds | val_loss | val_accuracy |\n")
    f.write("|---|---|---|---|---|---|\n")
    for _, row in gpu_results_df.iterrows():
        f.write(
            f"| {int(row['hidden_units'])} | {row['learning_rate']} | {int(row['epochs'])} | "
            f"{row['train_loop_seconds']:.6f} | {row['val_loss']:.4f} | {row['val_accuracy']:.4f} |\n"
        )
    f.write(f"\n**Mejor combinacion:** hidden={BEST_HIDDEN}, lr={BEST_LR}, epochs={BEST_EPOCHS}\n\n")

    f.write("## Entrenamiento final (GPU)\n\n")
    f.write(f"- **Transferencia host->device:** {H2D_SECONDS:.6f} s\n")
    f.write(f"- **Tiempo de entrenamiento (bucle de epocas):** {GPU_TRAIN_TIME_SEC:.6f} s\n")
    f.write(f"- **Muestras de entrenamiento:** leidas de `{os.path.basename(TRAIN_CSV)}`\n")
    f.write(f"- **Epocas:** {BEST_EPOCHS}\n")
    f.write(f"- **block_size:** {FINAL_BLOCK_SIZE}\n\n")

    f.write("## Efecto del tamaño de bloque\n\n")
    f.write("| block_size | train_loop_seconds | val_accuracy |\n")
    f.write("|---|---|---|\n")
    for _, row in block_size_df.iterrows():
        f.write(f"| {int(row['block_size'])} | {row['train_loop_seconds']:.6f} | {row['final_val_accuracy']:.4f} |\n")
    f.write("\n")

    f.write("## Monitoreo nvidia-smi\n\n")
    if gpu_samples:
        f.write(f"- **Muestras capturadas:** {len(gpu_samples)} (cada {POLL_INTERVAL_SEC}s, corrida con "
                f"`epochs={MONITOR_EPOCHS}` solo para esta medicion)\n")
        f.write(f"- **Utilizacion GPU:** max={GPU_UTIL_MAX:.1f}%, promedio={GPU_UTIL_AVG:.1f}%\n")
        f.write(f"- **Memoria GPU:** pico={GPU_MEM_PEAK_MB:.0f} MB de {GPU_MEM_TOTAL_MB:.0f} MB\n\n")
    else:
        f.write("- No se capturaron muestras: el entrenamiento real es demasiado corto para el "
                "intervalo de sondeo usado.\n\n")

    f.write("## Metricas en test\n\n")
    f.write("| Metrica | Valor |\n|---------|-------|\n")
    f.write(f"| Accuracy | {acc:.4f} |\n")
    f.write(f"| Precision | {prec:.4f} |\n")
    f.write(f"| Recall | {rec:.4f} |\n")
    f.write(f"| F1 | {f1:.4f} |\n\n")

    f.write("## Speedup GPU vs. CPU\n\n")
    if SPEEDUP is not None:
        f.write(f"- **CPU (`cpu_baseline_report.md`):** {CPU_TRAIN_TIME_SEC:.4f} s\n")
        f.write(f"- **GPU (este reporte):** {GPU_TRAIN_TIME_SEC:.4f} s\n")
        f.write(f"- **Speedup:** {SPEEDUP:.2f}x\n\n")
    else:
        f.write("No se encontro `cpu_baseline_report.md` con un tiempo de entrenamiento "
                "para calcular el speedup. Ejecuta primero `base-model-cpu.ipynb`.\n\n")

    f.write("## Uso\n\n")
    f.write(
        "Pesos finales en `weights_gpu.bin` (binario, leido por este notebook) y "
        "`weights_gpu.npz` (formato `app_streamlit`). Para usar este modelo como el "
        "modelo final del proyecto, copiar `weights_gpu.npz` a `modelo/weights.npz`.\n"
    )

print(f"Reporte guardado: {report_path}")

---
## Cierre

Con esto queda completa la Etapa 2: el modelo se entreno con kernels CUDA
propios, se midio el tiempo de entrenamiento (con su comparacion de speedup
contra la version CPU), el efecto del tamaño de bloque, y el uso de GPU con
`nvidia-smi`, segun lo pedido en la guia (seccion 6.3).

Si los resultados son satisfactorios, copiar manualmente `weights_gpu.npz` a
`modelo/weights.npz` para que sea el modelo que usa `app_streamlit/app.py`.